# Nuclear Enterprise 360 — Setup

This notebook imports the portable SQLite training database into Unity Catalog managed Delta tables.

**Safety boundary:** all data is synthetic. It does not represent a real nuclear facility, reactor, safety system, employee, procedure, or operational condition.

## Before running
1. Attach this notebook to **Serverless** compute.
2. Run the first setup cell to create the schema and volume.
3. Upload `nuclear_enterprise_360.db` through **New → Add or upload data → Upload files to a volume**.
4. Select the volume created by this notebook and rerun from the upload-check cell.

In [0]:
from pathlib import Path
import sqlite3
import pandas as pd
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "v2_nuclear_enterprise_360"
VOLUME = "training_files"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DB_PATH = f"{VOLUME_PATH}/nuclear_enterprise_360_v2_2_clean.db"

print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"Upload the SQLite file to: {VOLUME_PATH}")
print(f"Expected file: {DB_PATH}")

Catalog: workspace
Schema:  v2_nuclear_enterprise_360
Upload the SQLite file to: /Volumes/workspace/v2_nuclear_enterprise_360/training_files
Expected file: /Volumes/workspace/v2_nuclear_enterprise_360/training_files/nuclear_enterprise_360_v2_2_clean.db


In [0]:
VOLUME_PATH

'/Volumes/workspace/v2_nuclear_enterprise_360/training_files'

## Upload check
If this cell reports that the file is missing, use the upload instructions above. Databricks Free Edition restricts outbound downloads, so UI upload is the reliable method.

In [0]:
if not Path(DB_PATH).exists():
    raise FileNotFoundError(
        f"Upload nuclear_enterprise_360.db to {VOLUME_PATH}, then rerun this cell."
    )
print(f"Found training database: {DB_PATH}")

Found training database: /Volumes/workspace/v2_nuclear_enterprise_360/training_files/nuclear_enterprise_360_v2_2_clean.db


## Convert SQLite tables to managed Delta tables

The import is repeatable: rerunning it overwrites the training tables with their original synthetic state.

In [0]:
with sqlite3.connect(DB_PATH) as con:
    table_names = [
        row[0]
        for row in con.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        ).fetchall()
    ]

    import_results = []
    for table_name in table_names:
        pdf = pd.read_sql_query(f'SELECT * FROM "{table_name}"', con)
        pdf = pdf.astype(object).where(pd.notnull(pdf), None)
        sdf = spark.createDataFrame(pdf)
        target = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
        (
            sdf.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target)
        )
        import_results.append((table_name, len(pdf), len(pdf.columns)))

display(spark.createDataFrame(import_results, ["table_name", "row_count", "column_count"]))

table_name,row_count,column_count
a001_source_document_links,18,7
a001_source_documents,9,10
actions,807,10
agent_runs,50,9
agent_tool_calls,90,8
asset_health_scores,11520,6
assets,128,9
dataset_metadata,50,2
departments,8,5
document_chunks,59,6


## Add lakehouse features

The RAG source table uses Change Data Feed so it can later back a Delta Sync AI Search index.

In [0]:
# ──────────────────────────────────────────────────────────────────────
# Section 1: Enable Change Data Feed on the RAG source table
# ──────────────────────────────────────────────────────────────────────
# WHY: document_chunks is the source table that will later back a Vector Search
# index for the RAG pipeline.  Turning on Delta Change Data Feed (CDF) means
# every row-level insert / update / delete is captured in a hidden change log.
# This lets a Delta Sync AI Search index incrementally sync only the rows that
# changed (instead of a full rebuild every time), keeping the index fresh and
# inexpensive as documents are added or updated.
# ──────────────────────────────────────────────────────────────────────
spark.sql("ALTER TABLE document_chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# ──────────────────────────────────────────────────────────────────────
# Section 2: asset_360 — a single-row-per-asset "360-degree" business view
# ──────────────────────────────────────────────────────────────────────
# WHY: Analysts and dashboards need a denormalised, one-stop picture of every
# asset that fuses together information scattered across five base tables
# (assets, systems, asset_health_scores, work_orders, inspections).  Instead
# of re-joining all those tables in every downstream query, we define the
# logic once here and let consumers simply SELECT * FROM asset_360.
# ──────────────────────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE VIEW asset_360 AS
WITH latest_health AS (
  -- An asset may have many health-score snapshots over time.  We rank them
  -- by score_date descending and keep only the most recent one (rn = 1) so
  -- the view always reflects the *current* health picture.
  SELECT *, ROW_NUMBER() OVER (PARTITION BY asset_id ORDER BY score_date DESC) AS rn
  FROM asset_health_scores
), open_work AS (
  -- Aggregate *still-open* work orders per asset: total count plus a
  -- sub-count of HIGH/URGENT priority items so managers can quickly spot
  -- assets that need immediate attention.
  SELECT asset_id,
         COUNT(*) AS open_work_orders,
         SUM(CASE WHEN priority IN ('HIGH','URGENT') THEN 1 ELSE 0 END) AS high_priority_open_work
  FROM work_orders
  WHERE status IN ('OPEN','IN_PROGRESS','DEFERRED')
  GROUP BY asset_id
), findings AS (
  -- Count inspections that flagged a follow-up requirement, and capture the
  -- most recent such inspection date so the team knows how stale the finding is.
  SELECT asset_id,
         COUNT(*) AS follow_up_findings,
         MAX(inspection_date) AS latest_inspection_date
  FROM inspections
  WHERE follow_up_required = 1
  GROUP BY asset_id
)
SELECT a.asset_id, a.asset_name, a.asset_type, a.criticality, a.status,
       s.system_name, s.unit_id,
       h.health_score, h.risk_level, h.recommended_action,
       COALESCE(w.open_work_orders, 0) AS open_work_orders,
       COALESCE(w.high_priority_open_work, 0) AS high_priority_open_work,
       COALESCE(f.follow_up_findings, 0) AS follow_up_findings,
       f.latest_inspection_date
FROM assets a
JOIN systems s ON a.system_id = s.system_id
LEFT JOIN latest_health h ON a.asset_id = h.asset_id AND h.rn = 1
LEFT JOIN open_work w ON a.asset_id = w.asset_id
LEFT JOIN findings f ON a.asset_id = f.asset_id
""")

# ──────────────────────────────────────────────────────────────────────
# Section 3: project_risk_360 — one-row-per-project risk dashboard view
# ──────────────────────────────────────────────────────────────────────
# WHY: Project managers and executives need a quick risk summary per project.
# This view joins projects, risks, and actions, then rolls up key KPIs: how
# many risks are still open, the highest exposure among those risks, how many
# corrective actions are active, and how many are overdue.  As with asset_360,
# the goal is to pre-package the join + aggregation logic so every downstream
# dashboard or query gets consistent numbers without duplicating the logic.
# ──────────────────────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE VIEW project_risk_360 AS
SELECT p.project_id, p.project_name, p.status AS project_status,
       p.completion_pct, p.budget_usd,
       COUNT(DISTINCT CASE WHEN r.status IN ('OPEN','MITIGATING') THEN r.risk_id END) AS active_risks,
       MAX(CASE WHEN r.status IN ('OPEN','MITIGATING') THEN r.exposure_score END) AS maximum_exposure,
       COUNT(DISTINCT CASE WHEN a.status IN ('OPEN','IN_PROGRESS','OVERDUE') THEN a.action_id END) AS active_actions,
       COUNT(DISTINCT CASE WHEN a.status = 'OVERDUE' THEN a.action_id END) AS overdue_actions
FROM projects p
LEFT JOIN risks r ON p.project_id = r.project_id
LEFT JOIN actions a ON p.project_id = a.project_id
GROUP BY p.project_id, p.project_name, p.status, p.completion_pct, p.budget_usd
""")

print("Created asset_360 and project_risk_360 views.")

Created asset_360 and project_risk_360 views.


## Validation
The expected result is zero orphaned records and visible synthetic-data metadata.

In [0]:
# ──────────────────────────────────────────────────────────────────────
# Referential-integrity validation
# ──────────────────────────────────────────────────────────────────────
# After importing the SQLite tables into Unity Catalog, we need to verify that
# no foreign-key relationships were broken during the conversion. Each query
# uses a LEFT ANTI JOIN, which returns only the rows in the left table that do
# NOT have a matching key in the right table — i.e. orphaned records.
#
# The three checks performed:
#   1. work_orders_without_asset — work orders whose asset_id does not exist
#      in the assets table (every work order should be tied to a real asset).
#   2. risks_without_project — risks whose project_id does not exist in the
#      projects table (every risk should belong to a real project).
#   3. chunks_without_document — document chunks whose document_id does not
#      exist in the documents table (every chunk must belong to a real doc).
#
# If the synthetic data is clean, all three counts should be zero.
# ──────────────────────────────────────────────────────────────────────
validation = spark.sql("""
SELECT 'work_orders_without_asset' AS check_name, COUNT(*) AS issue_count
FROM work_orders w LEFT ANTI JOIN assets a ON w.asset_id = a.asset_id
UNION ALL
SELECT 'risks_without_project', COUNT(*)
FROM risks r LEFT ANTI JOIN projects p ON r.project_id = p.project_id
UNION ALL
SELECT 'chunks_without_document', COUNT(*)
FROM document_chunks c LEFT ANTI JOIN documents d ON c.document_id = d.document_id
""")

# Display the orphan counts — every row should show an issue_count of 0.
display(validation)

# Display the dataset_metadata table so we can confirm the synthetic data
# provenance info (e.g. description, generation date, safety notice) was
# imported correctly alongside the business tables.
display(spark.table("dataset_metadata"))

check_name,issue_count
work_orders_without_asset,0
risks_without_project,0
chunks_without_document,0


metadata_key,metadata_value
dataset_name,Nuclear Enterprise 360 AI Training Dataset
dataset_version,1.0.0
generated_on,2026-08-25
random_seed,3602026
data_classification,SYNTHETIC TRAINING DATA
safety_boundary,"No real facility, reactor, safety-system, employee, procedure, or operational data."
permitted_use,"Education, demonstrations, SQL, Delta, RAG, AI agents, and evaluation."
prohibited_interpretation,"Not valid for operational, engineering, regulatory, or safety decisions."
v2_extension_version,2.0.0
v2_created_on,2026-09-16


## Setup complete
Continue to `01_SQL_and_Delta_Foundations`.